#  How to Explore an Unknown Dataset - Quickstart

When exploring a linked dataset via a SPARQL endpoint for the first time, the hurdle can be very high and prickly. Unless one has prior knowledge of the structure of the ontology in question, or documentation is available to explain it in detail, one doesn't know where to turn. Luckily, there are strategies to start with when exploring a dataset. Discovering them is precisely the purpose of this workshop.

## The Structure of a Dataset
In general, the triples of a Linked Data source can be divided into two groups: A-Box and T-Box.
- The T-Box (Terminological Box) contains information related to the definition of classes, properties and more generally to the structure of the dataset. 
- The A-Box (Assertional Box), on the other hand, contains information about instances, relationships between instances and more generally information about the main content of the dataset.

### Note on the method
We will begin by exploring the T-Box. This way we can understand the size of the dataset and its descriptive structure. To do this, we can look at the number of triples present and the types of classes and properties used. We'll then examine in more detail the nature of the information contained in the dataset in relation to these classes and properties (A-Box).

The reference resource is the Zeri Foundation's SPARQL endpoint. Learn more [here](https://data.fondazionezeri.unibo.it/).

To run SPARQL queries from Python, we’ll use the `SPARQLWrapper` library in combination with `pandas`. This gives us a clean and reusable way to send queries and view the results as dataframes.

We define a helper function that:
1. Connects to the endpoint,
2. Sends the query and retrieves results in JSON format,
3. Converts the results to a pandas dataframe.

This setup allows us to inspect the structure and content of the dataset directly in our Python environment.

**Tip**: Before running queries in code, it's often helpful to test them directly in the SPARQL endpoint’s interface (if available). This helps debug syntax or logic issues more easily.

### Phase 1: Get Oriented – “What’s here?”
#### How many triples are there?
*Goal*: Understand dataset size.

In [6]:
# Uncomment if running in Colab or other clean environment
# !pip install SPARQLWrapper

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# Define endpoint
endpoint_url = "http://data.fondazionezeri.unibo.it/sparql"

# Define a function to query and return a DataFrame
def run_query(query, endpoint=endpoint_url):
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    # Normalize results to pandas dataframe
    cols = results["head"]["vars"]
    data = []
    for result in results["results"]["bindings"]:
        row = [result.get(col, {}).get("value", None) for col in cols]
        data.append(row)

    return pd.DataFrame(data, columns=cols)

In [7]:
query_triple_count = """
SELECT (COUNT(*) AS ?tripleCount)
WHERE {
  ?s ?p ?o .
}
"""

df = run_query(query_triple_count)
print("Total number of triples:")
print(df)

Total number of triples:
  tripleCount
0    11992769


#### What are the predicates used?
This is the quickest way to get an idea of the kind of data available. Many of the predicates can indeed tell us interesting things.
An RDF dataset may or may not have an explicit structure, and the use for example of a property such as rdfs:subClassOf can indicate its presence. The next query might then ask which classes are subclasses of which classes, so you can get an overview of the structure of the dataset. Or you can simply search for classes that are present.

*Goal*: Identify vocabulary used and maybe spot reused ontologies.

In [9]:
query_predicates = '''
    SELECT DISTINCT ?p
    WHERE { 
    ?s ?p ?o .
    }
'''

df = run_query(query_predicates)
print(f'The list of predicates:\n {df}')

The list of predicates:
                                                      p
0    http://purl.org/emmedi/hico/hasInterpretationC...
1    http://purl.org/emmedi/hico/hasInterpretationType
2          http://purl.org/emmedi/hico/isExtractedFrom
3              http://purl.org/spar/fabio/hasPortrayal
4            http://purl.org/spar/fabio/hasSubjectTerm
..                                                 ...
120               http://purl.org/dc/terms/description
121                 http://purl.org/dc/terms/publisher
122                     http://purl.org/dc/terms/title
123                    http://rdfs.org/ns/void#feature
124             http://rdfs.org/ns/void#sparqlEndpoint

[125 rows x 1 columns]


It could be interesting to try to figure out which properties are repeated most times. It is possible to do this by using the COUNT construct, and sorting the results in descending order (DESC).

##### Which predicates are used the most?
*Goal*: Spot central predicates (e.g., metadata, links, labels).

In [20]:
query_predicate_repetition = '''
    SELECT ?p (COUNT(?p) AS ?count)
    WHERE { 
    ?s ?p ?o .
    }
    GROUP BY ?p
    ORDER BY DESC(?count)
    LIMIT 10
'''

df = run_query(query_predicate_repetition)
print(f'The number of times each predicate is used:\n {df}')

HTTPError: HTTP Error 504: Gateway Time-out

It will certainly be interesting to delve into the 4/5 most recurrent properties later on. However, if you quickly look at the full list, you'll be able understand at a first glance how the information contained is mainly about the description of cultural objects and interaction with entities (artists/institutions).

### Phase 2: Explore the Structure (T-Box) – “What’s the schema?”
#### What classes are explicitly declared?
In RDF and OWL ontologies, classes can be defined explicitly using `rdfs:Class` or `owl:Class`. However, in many datasets, the actual instances are linked to these classes using `rdf:type`. Therefore, looking for all used `rdf:type` values often reveals more about the real content of the dataset than just exploring declared classes.
 Let's check with Zeri:

In [11]:
query_classes = '''
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?c
    WHERE {
        ?c a rdfs:Class .
    }
    ORDER BY ?c
'''
df = run_query(query_classes)
print(f'The list of classes:\n {df}')

The list of classes:
 Empty DataFrame
Columns: [c]
Index: []


In [12]:
query_classes = '''
    PREFIX owl: <http://www.w3.org/2002/07/owl#>
    SELECT DISTINCT ?c
    WHERE {
        ?c a owl:Class .
    }
    ORDER BY ?c
'''
df = run_query(query_classes)
print(f'The list of classes:\n {df}')

The list of classes:
 Empty DataFrame
Columns: [c]
Index: []


#### What classes are actually used in the data?
What we can do instead is to look at the concept that describes a subject, searching for its type: rdf:type or a.

*Goal*: Real-world usage of types — often richer than declared ontology.

In [13]:
query_classes = '''
    SELECT DISTINCT ?type 
    WHERE {
    ?s a ?type .
    }
'''
df = run_query(query_classes)
print(f'The list of Classes types:\n {df}')

The list of Classes types:
                                                   type
0      http://purl.org/spar/fabio/ExpressionCollection
1                    http://purl.org/spar/fabio/Thesis
2    http://www.essepuntato.it/2012/04/tvc/ValueInTime
3                      http://xmlns.com/foaf/0.1/Agent
4                      http://rdfs.org/ns/void#Dataset
..                                                 ...
100               http://purl.org/spar/fabio/Biography
101                 http://purl.org/spar/fabio/Bookset
102              http://purl.org/spar/fabio/BookSeries
103                  http://purl.org/spar/fabio/Review
104       http://purl.org/spar/fabio/InstructionalWork

[105 rows x 1 columns]


#### Most Common Classes by Instance Count
*Goal*: Understand which classes (types) are most used in the dataset.

This query counts how many distinct resources are typed as each class. Using `COUNT(DISTINCT ?s)` ensures we don’t overcount in case of duplicate type assertions.

If the query takes too long to run, it might be helpful to add a `LIMIT` or filter to specific namespaces.

In [24]:
query_common_classes= '''
    SELECT ?type (COUNT(DISTINCT ?s) AS ?instanceCount)
    WHERE {
    ?s a ?type .
    }
    GROUP BY ?type
    ORDER BY DESC(?instanceCount)
'''

df = run_query(query_common_classes)
print(f'Most common classes in descending order:\n {df}')

HTTPError: HTTP Error 504: Gateway Time-out

### Phase 3: Explore Content (A-Box) – “What’s inside?”
#### Properties Used for a Specific Class

In [21]:
query_class_properties = '''
    SELECT ?p (COUNT(*) AS ?count)
    WHERE {
    ?s a <http://example.org/ChosenType> ;
        ?p ?o .
    }
    GROUP BY ?p
    ORDER BY DESC(?count)
'''

df = run_query(query_class_properties)
print(f'The number of properties per class are:\n {df}')

KeyboardInterrupt: 

#### What are the labels for instances?
With Classes, and in particular the instances that refer to them, we get to the heart of the dataset's content. First we can look at some instances' labels.

*Goal*: Make resources more readable, support UI/data inspection.

In [ ]:
query_instance__label = '''
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?instance ?label
    WHERE {
    ?instance a ?type .
    OPTIONAL { ?instance rdfs:label ?label }
    }
    LIMIT 100
'''

df = run_query(query_instance__label)
print(f'The number of instances per class are:\n {df}')

The number of instances per class are:
                                              instance  \
0   https://w3id.org/zericatalog/photo/37095/photo...   
1   https://w3id.org/zericatalog/photo/37095/photo...   
2   https://w3id.org/zericatalog/photo/37096/attri...   
3   https://w3id.org/zericatalog/photo/37096/attri...   
4   https://w3id.org/zericatalog/photo/37096/attri...   
..                                                ...   
95  https://w3id.org/zericatalog/photo/37119/attri...   
96  https://w3id.org/zericatalog/photo/37119/attri...   
97  https://w3id.org/zericatalog/photo/37119/attri...   
98  https://w3id.org/zericatalog/photo/37119/photo...   
99  https://w3id.org/zericatalog/photo/37119/photo...   

                                     label  
0          Authorship attribution: Anonimo  
1     Attribuzione di autorialità: Anonimo  
2       Attribution of date: 1870ca-1940ca  
3   Attribuzione della data: 1870ca-1940ca  
4              Attribution of proper title  
..   

#### How many properties does each class use?
*Goal*: See how rich/descriptive each class is.

In [23]:
property_per_class = '''
    SELECT ?type (COUNT(DISTINCT ?p) AS ?propertyCount)
    WHERE {
    ?s a ?type ;
        ?p ?o .
    }
    GROUP BY ?type
    ORDER BY DESC(?propertyCount)
'''

df = run_query(property_per_class)
print(f'The number of instances per class are:\n {df}')

HTTPError: HTTP Error 504: Gateway Time-out

If we go back and look at the properties that occur most often, we find `rdfs:label` frequently at the top. This is a very useful property that allows instances to be named in natural language.

However, a common issue is duplicate labels that differ slightly, often due to typos or inconsistent formatting (e.g., "Federico Zeri" vs. " Federico Zeri" with an extra space). These are treated as different strings, even though they refer to the same concept.

One strategy to partially handle this is using the `SAMPLE()` function, which **selects a single representative label** for each instance or group. However, note that this doesn’t merge or clean similar labels — it just picks one of them.

In [26]:
query_instance_label = '''
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 
    SELECT ?instance 
        (SAMPLE(?label) AS ?instanceLabel) 
        (COUNT(?instance) AS ?instanceCount) 
    WHERE { 
        ?instance a ?class . 
        OPTIONAL{ ?instance rdfs:label ?label .} 
        }
        GROUP BY ?instance ?instanceLabel
        ORDER BY DESC(?instanceCount)
'''

df = run_query(query_instance_label)
print(f'The list of instances with labels and repetitions:\n {df}')

HTTPError: HTTP Error 504: Gateway Time-out

# References

- DuCharme Bob, «Exploring a SPARQL endpoint», 24 agosto 2014. https://www.bobdc.com/blog/exploring-a-sparql-endpoint/.
- DuCharme Bob, «Queries to explore a dataset», 30 aprile 2022. https://www.bobdc.com/blog/exploringadataset/.